In [1]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, load_dataset, Features, Value
from transformers import (
    AlbertTokenizer,
    AlbertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import torch

In [2]:
# Download data from huggingface repositiory (we uploaded the data first).
file_path = hf_hub_download(
    repo_id="MikkelPraestegaard/GDS_final_assignment",
    filename="subset_FakeNews.zip",
    repo_type="dataset",
)

In [3]:
# Check cuda
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA GeForce GTX 1650 Ti


In [ ]:
seed = 123

# Load data
dataset = load_dataset(
    "csv",
    data_files=file_path,
    delimiter=",",
    encoding="utf-8",
    on_bad_lines="skip",
    features=Features({"content": Value("string"), "type": Value("string")})
)["train"]

# Remove NaNs from content column.
dataset = dataset.filter(lambda x: x["content"] is not None and x["content"] != "")

# Remove rows with ambiguous True or False type (eg. satire).
ambiguous = ['unknown', 'nan', 'political', 'clickbait', '2018-02-10 13:43:39.521661']


dataset = dataset.filter(lambda x: x["type"] not in ambiguous)

# Create new column for True or Fake labels. True will be labeled 0 and fake will be labeled 1.
true_list = ['reliable']

def create_target(data):
    data["labels"] = int(data["type"] not in true_list)
    return data

dataset = dataset.map(create_target)

Map:   0%|          | 0/929515 [00:00<?, ? examples/s]

In [5]:
# Split data
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=seed
)

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_dataset = temp_dataset.train_test_split(
    test_size=0.5,
    seed=seed
)

val_dataset = temp_dataset["train"]
test_dataset = temp_dataset["test"]

In [6]:
train_dataset['type']

Column(['reliable', None, 'reliable', 'unreliable', 'political', ...])

In [7]:
# Load tokenizer
tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, max_length=128)
# Tokenize
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Remove unused columns
train_dataset = train_dataset.remove_columns(["content"])
val_dataset = val_dataset.remove_columns(["content"])

# Set datasets to pytorch format
train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/743612 [00:00<?, ? examples/s]

Map:   0%|          | 0/92951 [00:00<?, ? examples/s]

Map:   0%|          | 0/92952 [00:00<?, ? examples/s]

In [8]:
model = AlbertForSequenceClassification.from_pretrained('albert-base-v2', num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    f1 = f1_score(labels, predictions)

    return {'f1': f1}


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/albert_results",
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,             # only keep the 2 most recent checkpoints to save disk space
    logging_strategy="steps",
    logging_steps=2000,
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
    dataloader_num_workers=4        # parallel data loading
)
# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [10]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.103868,0.108373,0.983235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
# Save model and tokenizer
trainer.save_model("./fine_tuned_albert")
tokenizer.save_pretrained("./fine_tuned_albert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_albert/tokenizer_config.json',
 './fine_tuned_albert/tokenizer.json')